In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# Project 17 — Local Personal Wiki Copilot
## Query Exported Notes/Wiki Files Locally

**Stack:** Ollama · LlamaIndex · Jupyter

In [ ]:
# !pip install -q llama-index llama-index-llms-ollama llama-index-embeddings-ollama

## Step 1 — Setup

In [ ]:
from llama_index.core import Settings, VectorStoreIndex, Document
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding

Settings.llm = Ollama(model="qwen3:8b", request_timeout=120.0)
Settings.embed_model = OllamaEmbedding(model_name="nomic-embed-text")

## Step 2 — Create Sample Wiki

In [ ]:
wiki_pages = [
    Document(text="""# Project Phoenix
Status: Active | Lead: Alice | Started: 2024-Q3

## Goals
- Migrate legacy monolith to microservices
- Reduce deployment time from 4 hours to 15 minutes
- Achieve 99.95% uptime SLA

## Architecture Decisions
- ADR-001: Use Kubernetes for orchestration (approved)
- ADR-002: Use PostgreSQL over MySQL (approved)
- ADR-003: Event-driven architecture with Kafka (in review)

## Current Status
Phase 1 (Auth service) complete. Phase 2 (Payment service) in progress.
Blocked by: Legacy database schema migration.""",
    metadata={"page": "Project Phoenix", "type": "project", "tags": "architecture,migration"}),

    Document(text="""# Onboarding Checklist
## Week 1
- [ ] Set up dev environment (see Dev Setup page)
- [ ] Complete security training
- [ ] Get access to GitHub, Jira, Slack
- [ ] Meet with team lead and buddy

## Week 2
- [ ] Complete first PR (good-first-issue label)
- [ ] Shadow a production deployment
- [ ] Read Architecture Decision Records

## Tools
- IDE: VS Code with recommended extensions
- Terminal: Use Oh My Zsh with our custom theme
- VPN: Required for staging/production access""",
    metadata={"page": "Onboarding", "type": "process", "tags": "hr,new-hire"}),

    Document(text="""# Incident Response Playbook
## Severity Levels
- SEV1: Customer-facing outage, all hands on deck
- SEV2: Degraded service, team lead + on-call
- SEV3: Internal tooling issue, on-call engineer

## Response Steps
1. Acknowledge the alert within 5 minutes
2. Create incident channel: #inc-YYYY-MM-DD-brief-description
3. Assign Incident Commander and Communications Lead
4. Begin investigation, post updates every 15 min for SEV1
5. After resolution: blameless postmortem within 48 hours

## Escalation
On-call → Team Lead → Engineering Director → CTO""",
    metadata={"page": "Incident Response", "type": "playbook", "tags": "ops,incidents"}),
]
print(f"Created {len(wiki_pages)} wiki pages")

## Step 3 — Build Wiki Index

In [ ]:
index = VectorStoreIndex.from_documents(wiki_pages, show_progress=True)
query_engine = index.as_query_engine(similarity_top_k=3)
print("Wiki search index ready!")

## Step 4 — Query the Wiki

In [ ]:
queries = [
    "What's the current status of Project Phoenix?",
    "What do I need to do in my first week as a new hire?",
    "What's the process for a SEV1 incident?",
    "What database did we choose and why?",
]

for q in queries:
    print(f"\nQ: {q}")
    response = query_engine.query(q)
    print(f"A: {response}")
    pages = [n.metadata.get('page', '?') for n in response.source_nodes]
    print(f"Found in: {pages}")

## Step 5 — Wiki Chat with Context

In [ ]:
from llama_index.core.chat_engine import CondenseQuestionChatEngine

chat = CondenseQuestionChatEngine.from_defaults(query_engine=query_engine)

conversation = [
    "Tell me about our architecture decisions",
    "Which one is still in review?",
    "What event system are we considering?",
]
for msg in conversation:
    print(f"\nUser: {msg}")
    resp = chat.chat(msg)
    print(f"Wiki: {resp}")

## What You Learned
- **Wiki page indexing** with metadata tags
- **Cross-page search** for internal knowledge
- **Conversational wiki** with context memory